<a href="https://colab.research.google.com/github/edgargonarr/Matem-ticas-Discretas-Notas/blob/Ejemplo-de-rama/Proyectito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# SVR Model for Aircraft Engine Performance Evaluation
# Based on: "Evaluation of aircraft engine performance during takeoff phase
#            with machine learning methods" (Kurt, 2024)
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =============================================================================
# 1. CARGA DE DATOS
# =============================================================================

# Cargar el dataset desde el archivo CSV
# El archivo debe contener las 8 columnas: 7 inputs + 1 target
df = pd.read_csv('EEDB_data.csv')

print("=" * 60)
print("CARGA DE DATOS")
print("=" * 60)
print(f"Shape del dataset: {df.shape}")
print(f"\nPrimeras 5 filas:\n{df.head()}")
print(f"\nEstadísticas descriptivas:\n{df.describe()}")

# =============================================================================
# 2. DEFINICIÓN DE VARIABLES
# =============================================================================

# Variables de entrada (features) según el artículo (Tabla 4)
feature_columns = [
    'Ambient_humidity',  # Humedad ambiental (kg/kg)
    'Engine_type',       # Tipo de motor (categórica: 1 o 2)
    'Ambient_baro',      # Presión barométrica ambiental (kPa)
    'Rated_output',      # Potencia nominal (kN)
    'Ambient_temp',      # Temperatura ambiental (K)
    'Bypass_ratio',      # Relación de derivación
    'Press_ratio'        # Relación de presión
]

# Variable objetivo (target): Flujo de combustible en despegue (kg/s)
target_column = 'Fuel_flow_TO'

# Separar features y target
X = df[feature_columns].values
y = df[target_column].values

print("\n" + "=" * 60)
print("VARIABLES")
print("=" * 60)
print(f"Features (X): {feature_columns}")
print(f"Target (y):   {target_column}")
print(f"\nShape X: {X.shape}")
print(f"Shape y: {y.shape}")

# =============================================================================
# 3. PREPROCESAMIENTO: Estandarización y División del Dataset
# =============================================================================

# --- División del dataset ---
# Paso 1: Separar 70% train y 30% temporal (val + test)
# Paso 2: Dividir el 30% temporal en 50/50 → 15% val y 15% test
# Se usa random_state=42 para reproducibilidad

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,        # 30% para val + test
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,        # 50% del 30% = 15% del total para test
    random_state=42
)

print("\n" + "=" * 60)
print("DIVISIÓN DEL DATASET")
print("=" * 60)
print(f"Total muestras    : {len(X)}")
print(f"Train (70%)       : {len(X_train)}")
print(f"Validación (15%)  : {len(X_val)}")
print(f"Test (15%)        : {len(X_test)}")

# --- Estandarización (StandardScaler) ---
# IMPORTANTE: El scaler se ajusta SOLO con los datos de entrenamiento
# para evitar data leakage en validación y test.
# StandardScaler transforma: z = (x - mean) / std

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Ajusta y transforma train
X_val_scaled   = scaler.transform(X_val)          # Solo transforma (sin re-ajustar)
X_test_scaled  = scaler.transform(X_test)         # Solo transforma (sin re-ajustar)

print("\n" + "=" * 60)
print("ESTANDARIZACIÓN (StandardScaler)")
print("=" * 60)
print(f"Media de features (calculada en train):\n{scaler.mean_.round(4)}")
print(f"\nDesv. estándar (calculada en train):\n{scaler.scale_.round(4)}")

# =============================================================================
# 4. MODELO SVR - HIPERPARÁMETROS EXACTOS DEL ARTÍCULO
# =============================================================================
# Según el artículo (Tabla 13 - Quadratic SVM):
#   - kernel   : 'poly'   → kernel polinómico
#   - degree   : 2        → grado 2 (cuadrático)
#   - gamma    : 'auto'   → 1 / n_features
#   - C        : 1.2076   → Box constraint (parámetro de penalización)
#   - epsilon  : 0.1208   → Tubo epsilon-insensible (margen de tolerancia)

svr_model = SVR(
    kernel='poly',
    degree=2,
    gamma='auto',
    C=1.2076,
    epsilon=0.1208
)

print("\n" + "=" * 60)
print("CONFIGURACIÓN DEL MODELO SVR (Quadratic SVM)")
print("=" * 60)
print(f"  kernel  : {svr_model.kernel}")
print(f"  degree  : {svr_model.degree}")
print(f"  gamma   : {svr_model.gamma}")
print(f"  C       : {svr_model.C}")
print(f"  epsilon : {svr_model.epsilon}")

# =============================================================================
# 5. ENTRENAMIENTO
# =============================================================================

print("\n" + "=" * 60)
print("ENTRENAMIENTO DEL MODELO")
print("=" * 60)
print("Entrenando SVR con datos de train estandarizados...")

svr_model.fit(X_train_scaled, y_train)

print("✓ Entrenamiento completado.")
print(f"  Vectores de soporte encontrados: {svr_model.n_support_}")

# =============================================================================
# 6. PREDICCIONES
# =============================================================================

# Generar predicciones para validación y test
y_pred_train = svr_model.predict(X_train_scaled)
y_pred_val   = svr_model.predict(X_val_scaled)
y_pred_test  = svr_model.predict(X_test_scaled)

# =============================================================================
# 7. MÉTRICAS DE EVALUACIÓN
# =============================================================================

def calculate_mape(y_true, y_pred):
    """
    MAPE (Mean Absolute Percentage Error):
    Mide el error promedio porcentual absoluto.
    Se excluyen valores de y_true = 0 para evitar división por cero.
    MAPE < 10% → Alta precisión (Lewis, 1982)
    """
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_model(y_true, y_pred, set_name=""):
    """Calcula e imprime MSE, MAE, MAPE y R² para un conjunto dado."""
    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    r    = np.sqrt(abs(r2)) if r2 >= 0 else 0   # Correlación R

    print(f"\n  {'─'*40}")
    print(f"  Conjunto: {set_name}")
    print(f"  {'─'*40}")
    print(f"  MSE  : {mse:.7f}")
    print(f"  MAE  : {mae:.7f}")
    print(f"  MAPE : {mape:.6f}%")
    print(f"  R²   : {r2:.5f}")
    print(f"  R    : {r:.5f}")

    return {'MSE': mse, 'MAE': mae, 'MAPE': mape, 'R2': r2, 'R': r}


print("\n" + "=" * 60)
print("RESULTADOS DE EVALUACIÓN")
print("=" * 60)

metrics_train = evaluate_model(y_train, y_pred_train, "Entrenamiento (Train)")
metrics_val   = evaluate_model(y_val,   y_pred_val,   "Validación (Val)")
metrics_test  = evaluate_model(y_test,  y_pred_test,  "Prueba (Test)")

# =============================================================================
# 8. COMPARACIÓN CON RESULTADOS DEL ARTÍCULO
# =============================================================================

print("\n" + "=" * 60)
print("COMPARACIÓN CON RESULTADOS DEL ARTÍCULO (Tabla 12 y 16)")
print("=" * 60)
print(f"{'Métrica':<10} {'Artículo':>12} {'Este script':>12}")
print("-" * 36)
print(f"{'R (train)':<10} {'0.99655':>12} {metrics_train['R']:>12.5f}")
print(f"{'R (val)':<10} {'0.97499':>12} {metrics_val['R']:>12.5f}")
print(f"{'R (test)':<10} {'0.99709':>12} {metrics_test['R']:>12.5f}")
print(f"{'MSE (test)':<10} {'0.0054221':>12} {metrics_test['MSE']:>12.7f}")
print(f"{'MAE (test)':<10} {'0.059196':>12} {metrics_test['MAE']:>12.6f}")
print(f"{'MAPE (test)':<10} {'4.909031%':>12} {metrics_test['MAPE']:>11.6f}%")

# =============================================================================
# 9. CLASIFICACIÓN SEGÚN LEWIS (1982)
# =============================================================================

print("\n" + "=" * 60)
print("CLASIFICACIÓN DEL MODELO (Lewis, 1982)")
print("=" * 60)

def classify_mape(mape_val):
    if mape_val < 10:
        return "✓ MUY BUENO (MAPE < 10%)"
    elif mape_val < 20:
        return "✓ BUENO (10% ≤ MAPE < 20%)"
    elif mape_val < 50:
        return "⚠ RAZONABLE (20% ≤ MAPE < 50%)"
    else:
        return "✗ INEXACTO (MAPE ≥ 50%)"

print(f"  Test MAPE: {metrics_test['MAPE']:.4f}%  →  {classify_mape(metrics_test['MAPE'])}")

print("\n" + "=" * 60)
print("SCRIPT FINALIZADO EXITOSAMENTE")
print("=" * 60)

CARGA DE DATOS
Shape del dataset: (500, 8)

Primeras 5 filas:
   Ambient_humidity  Engine_type  Ambient_baro  Rated_output  Ambient_temp  \
0          0.014982            1       97.4359        303.43        256.07   
1          0.012843            2      100.4706        103.73        297.36   
2          0.007205            2       99.8821         60.04        312.93   
3          0.000162            1       92.0007        338.13        235.55   
4          0.012406            1       91.9402        134.13        304.94   

   Bypass_ratio  Press_ratio  Fuel_flow_TO  
0        7.4908      32.9770      8.192182  
1        0.3168      13.7047      4.703363  
2        0.4492      23.5938      3.769758  
3        8.0690      28.5312      9.164033  
4        7.4237      29.5049      4.676650  

Estadísticas descriptivas:
       Ambient_humidity  Engine_type  Ambient_baro  Rated_output  \
count        500.000000   500.000000    500.000000    500.000000   
mean           0.012522     1.30800

In [ ]:
# ============================================================================
# MÓDULO: generate_EEDB_data.py
# Genera EEDB_data.csv con datos sintéticos físicamente coherentes
# para probar svr_aircraft_engine.py en Google Colab
# ============================================================================

import numpy as np
import pandas as pd
import os

# ── Reproducibilidad ─────────────────────────────────────────────────────────
np.random.seed(42)
N = 500  # Número de registros (ajustable)

print("=" * 60)
print("GENERADOR DE DATOS SINTÉTICOS - EEDB")
print("=" * 60)

# =============================================================================
# 1. ENGINE_TYPE  (categórica: 1 = Turbofan, 2 = Turbojet)
# =============================================================================
engine_type = np.random.choice([1, 2], size=N, p=[0.70, 0.30])

# =============================================================================
# 2. BYPASS_RATIO  (adimensional)
#    Turbofan : 4.0–9.0  |  Turbojet : 0.1–0.5
# =============================================================================
bypass_ratio = np.where(
    engine_type == 1,
    np.random.uniform(4.0, 9.0, N),
    np.random.uniform(0.1, 0.5, N)
)

# =============================================================================
# 3. PRESS_RATIO — Overall Pressure Ratio (adimensional)
#    Turbofan : 20–45  |  Turbojet : 10–25
# =============================================================================
press_ratio = np.where(
    engine_type == 1,
    np.random.uniform(20.0, 45.0, N),
    np.random.uniform(10.0, 25.0, N)
)

# =============================================================================
# 4. RATED_OUTPUT — Empuje nominal (kN)
#    Turbofan : 80–350 kN  |  Turbojet : 40–120 kN
# =============================================================================
rated_output = np.where(
    engine_type == 1,
    np.random.uniform(80.0, 350.0, N),
    np.random.uniform(40.0, 120.0, N)
)

# =============================================================================
# 5. CONDICIONES AMBIENTALES (variaciones ISA)
# =============================================================================
ambient_temp     = np.random.uniform(233.15, 313.15, N)  # K (-40°C a +40°C)
ambient_baro     = np.random.uniform(90.0, 103.0, N)     # kPa
ambient_humidity = np.random.uniform(0.0, 0.025, N)      # kg/kg

# =============================================================================
# 6. FUEL_FLOW_TO — Variable objetivo (kg/s)
#
# Fórmula semi-empírica con correcciones ISA (método Delta-Theta):
#   θ = T/T_ISA   δ = P/P_ISA
#   Mayor bypass  → mejor eficiencia → MENOS combustible
#   Aire más cálido o menor presión → MÁS combustible
# =============================================================================
theta = ambient_temp / 288.15   # Ratio de temperatura (ISA)
delta = ambient_baro / 101.325  # Ratio de presión (ISA)

base_fuel = (
    rated_output  * 0.025 +
    press_ratio   * 0.015 +
    (1.0 / (bypass_ratio + 1)) * 2.5
)

fuel_flow_TO = (
    base_fuel * (theta ** 0.5) / delta
    + ambient_humidity * 0.8
    + np.random.normal(0, 0.05, N)   # Ruido operacional
)
fuel_flow_TO = np.clip(fuel_flow_TO, 0.5, 12.0)  # Límites físicos (kg/s)

# =============================================================================
# 7. DATAFRAME  — columnas en el mismo orden que svr_aircraft_engine.py
# =============================================================================
df = pd.DataFrame({
    'Ambient_humidity': np.round(ambient_humidity, 6),
    'Engine_type':      engine_type.astype(int),
    'Ambient_baro':     np.round(ambient_baro, 4),
    'Rated_output':     np.round(rated_output, 2),
    'Ambient_temp':     np.round(ambient_temp, 2),
    'Bypass_ratio':     np.round(bypass_ratio, 4),
    'Press_ratio':      np.round(press_ratio, 4),
    'Fuel_flow_TO':     np.round(fuel_flow_TO, 6)
})

# =============================================================================
# 8. REPORTE
# =============================================================================
print(f"\n{'Columna':<20} {'Min':>8} {'Max':>8} {'Media':>8} {'Std':>8}")
print("-" * 56)
for col in df.columns:
    print(f"{col:<20} {df[col].min():>8.3f} {df[col].max():>8.3f} "
          f"{df[col].mean():>8.3f} {df[col].std():>8.3f}")

print(f"\nDistribución Engine_type:")
for etype, cnt in df['Engine_type'].value_counts().sort_index().items():
    label = "Turbofan" if etype == 1 else "Turbojet"
    print(f"  Tipo {etype} ({label}): {cnt} ({cnt/N*100:.0f}%)")

print(f"\nRegistros: {len(df)}  |  Nulos: {df.isnull().sum().sum()}")
print(f"\nPrimeras 5 filas:\n{df.head().to_string()}")

# =============================================================================
# 9. EXPORTAR CSV + DESCARGA AUTOMÁTICA EN COLAB
# =============================================================================
output_file = 'EEDB_data.csv'
df.to_csv(output_file, index=False)
print(f"\nArchivo guardado: {os.path.abspath(output_file)}")
print(f"Tamaño: {os.path.getsize(output_file)/1024:.1f} KB")

# Descarga automática solo si estamos en Google Colab
try:
    from google.colab import files
    files.download(output_file)
    print("Descarga iniciada en tu navegador.")
except ImportError:
    print("(Entorno local) Archivo disponible en el directorio actual.")

print("\nListo. Ahora ejecuta svr_aircraft_engine.py con este CSV.")

GENERADOR DE DATOS SINTÉTICOS - EEDB

Columna                   Min      Max    Media      Std
--------------------------------------------------------
Ambient_humidity        0.000    0.025    0.013    0.007
Engine_type             1.000    2.000    1.308    0.462
Ambient_baro           90.000  102.971   96.336    3.677
Rated_output           40.160  349.410  176.112   93.059
Ambient_temp          233.200  313.110  271.783   23.203
Bypass_ratio            0.105    8.999    4.543    3.060
Press_ratio            10.350   44.959   27.889    9.285
Fuel_flow_TO            2.757   10.693    5.783    2.031

Distribución Engine_type:
  Tipo 1 (Turbofan): 346 (69%)
  Tipo 2 (Turbojet): 154 (31%)

Registros: 500  |  Nulos: 0

Primeras 5 filas:
   Ambient_humidity  Engine_type  Ambient_baro  Rated_output  Ambient_temp  Bypass_ratio  Press_ratio  Fuel_flow_TO
0          0.014982            1       97.4359        303.43        256.07        7.4908      32.9770      8.192182
1          0.012843    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada en tu navegador.

Listo. Ahora ejecuta svr_aircraft_engine.py con este CSV.
